In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("pertemuan6") \
    .master("local[*]") \
    .getOrCreate() 
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi spark: ", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 10:12:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/01 10:12:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi spark:  3.5.9


In [8]:
# Membuat dataset berisi 20.000 baris 

np.random.seed(11)
n = 20000
data_besar = {
    "order_id": [f"O{i}" for i in range(n)],
    "customer_id": np.random.randint(1, 21, size=n),
    "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"], size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"], size=n),
    "metode_pembayaran": np.random.choice(["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
pd.DataFrame(data_besar).to_csv("data_transaksi_besar.csv", index=False)

csv_size = os.path.getsize("data_transaksi_besar.csv")
print(f"ukuran CSV: {csv_size:,} bytes ({csv_size/1024:.1f} KB) untuk {n:,} baris")

ukuran CSV: 892,909 bytes (872.0 KB) untuk 20,000 baris


In [18]:
# Membaca csv lalu tulis ulang sebagai parquet

df_besar = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_besar.write.mode("overwrite").parquet("data_transaksi_besar_parquet")

# Menghitung total ukuran seluruh berkas .parquet yang dihasilkan

parquet_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk("data_transaksi_besar_parquet")
    for f in fn if f.endswith(".parquet")
)

print(f"Ukuran CSV      : {csv_size:,} bytes ({csv_size/1024:.1f} KB)")
print(f"ukuran parquet: {parquet_size:,} bytes ({parquet_size/1024:.1f} KB)")
print(f"Parquet {(1 - parquet_size/csv_size)*100:.1f}% lebih kecil dibanding CSV")

Ukuran CSV      : 892,909 bytes (872.0 KB)
ukuran parquet: 202,197 bytes (197.5 KB)
Parquet 77.4% lebih kecil dibanding CSV


In [19]:
#Skema otomatis tersimpan ke parquet

df_dari_parquet = spark.read.parquet("data_transaksi_besar_parquet")
df_dari_parquet.printSchema()
df_dari_parquet.show(5)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)

+--------+-----------+----------+---------+-----------------+----------+
|order_id|customer_id|  kategori|     kota|metode_pembayaran|pendapatan|
+--------+-----------+----------+---------+-----------------+----------+
|      O0|         17|   Makanan|     Solo|              COD|    196650|
|      O1|         18| Kesehatan| Magelang|         E-Wallet|    283512|
|      O2|         14|Elektronik| Semarang|              COD|    108835|
|      O3|         13| Kesehatan|Purworejo|     Kartu Kredit|    390604|
|      O4|          2|   Fashion|Purworejo|         E-Wallet|    427224|
+--------+-----------+----------+---------+-----------------+----------+
only showing top 5 rows



In [20]:
import json

np.random.seed(21)
pelanggan = [
    {
        "customer_id": i,
        "nama": f"Pelanggan{i}",
        "membership": np.random.choice(["Silver", "Gold", "Platinum"]),
        "kota_domisili": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworwjo"])
    }
    for i in range(1, 21)
]

with open("data_pelanggan.json", "w") as f:
    for p in pelanggan:
        f.write(json.dumps(p) + "\n")

print("Berkas data_pelanggan.json berhasil dibuat (format JSON Lines).")

Berkas data_pelanggan.json berhasil dibuat (format JSON Lines).


In [21]:
df_pelanggan = spark.read.json("data_pelanggan.json")
df_pelanggan.printSchema()
df_pelanggan.show(5)

root
 |-- customer_id: long (nullable = true)
 |-- kota_domisili: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- nama: string (nullable = true)

+-----------+-------------+----------+----------+
|customer_id|kota_domisili|membership|      nama|
+-----------+-------------+----------+----------+
|          1|     Magelang|      Gold|Pelanggan1|
|          2|     Magelang|    Silver|Pelanggan2|
|          3|     Magelang|    Silver|Pelanggan3|
|          4|         Solo|    Silver|Pelanggan4|
|          5|         Solo|      Gold|Pelanggan5|
+-----------+-------------+----------+----------+
only showing top 5 rows

